# UNIDAD IV — Manejo de archivos, persistencia y validación

## CLASE 3: Validaciones, errores y control de excepciones

---

# PARTE I — FUNDAMENTOS TEÓRICOS (Comprensión profunda)



---



# 1. ¿Qué es un error realmente?

Un error NO es solo que el programa falle.

Es una ruptura del contrato entre:

```
Lo que el programa espera

vs

Lo que realmente ocurre
```

---

## Tipos de errores en ingeniería de software

| Tipo                | Momento           | Ejemplo            |
| ------------------- | ----------------- | ------------------ |
| Sintáctico          | Antes de ejecutar | falta de :         |
| Lógico              | Funciona mal      | cálculo incorrecto |
| Runtime (excepción) | Durante ejecución | archivo no existe  |

Esta clase trata el tercero.

---

# 2. ¿Qué es una excepción?

Es un mecanismo controlado de fallo.

Python detiene la ejecución normal y busca un manejador.

Ejemplo:

```python
int("hola")
```

Produce:

```
ValueError
```

El programa no sabe continuar.

---

# 3. Filosofía profesional

Programación principiante:

> evitar errores

Programación profesional:

> asumir que los errores SIEMPRE ocurrirán

Por eso existen las excepciones.

---

# 4. try / except

Estructura básica:


In [ ]:
try:
    codigo_peligroso
except TipoError:
    recuperacion


---

## Ejemplo pedagógico


In [ ]:
try:
    edad = int(input("Edad: "))
except ValueError:
    print("Debe ingresar un número")



No evita el error — lo controla.

---


# 5. Múltiples excepciones



In [ ]:
try:
    edad = int(input("Edad: "))
except ValueError:
    print("Debe ser un número válido.")
except TypeError:
    print("Tipo de dato incorrecto.")
except Exception as e:
    print(f"Ocurrió un error inesperado: {e}")

---

# 6. Capturar sin saber el tipo (mala práctica)

```python
except:
```

Problema:

Oculta bugs graves.

Regla profesional:

> Nunca capturar excepciones genéricas sin razón

Correcto:

```python
except Exception as e:
    print(e)
```

---


# 7. else y finally

```python
try:
    lectura
except Error:
    manejar
else:
    si_todo_salio_bien
finally:
    siempre_se_ejecuta
```

---

## Uso real

| Bloque  | Propósito               |
| ------- | ----------------------- |
| try     | operación peligrosa     |
| except  | recuperación            |
| else    | lógica posterior segura |
| finally | liberar recursos        |

---



In [ ]:
try:
    edad = int(input("Edad: "))
except ValueError:
    print("Debe ser un número válido.")
else:
    print(f"Edad ingresada correctamente: {edad}")
finally:
    print("Fin del proceso.")

# 8. Errores comunes en archivos

---

## FileNotFoundError

Archivo inexistente

## PermissionError

Archivo bloqueado por otro programa

## UnicodeDecodeError

Codificación incorrecta

## ValueError

Formato inválido

## KeyError

Columna inexistente en CSV/dict

---


# 9. Validar vs Manejar excepción

MUY IMPORTANTE:

No todo error debe atraparse.

Comparación:

| Estrategia    | Cuándo          |
| ------------- | --------------- |
| Validar antes | datos usuario   |
| Try/Except    | sistema externo |

---

Ejemplo:

MAL:

```python
try:
    edad = int(input())
except:
    pass
```

BIEN:

```python
dato = input()
if not dato.isdigit():
    print("Edad inválida")
```

---



# 10. Diseño de mensajes de error

El usuario no debe ver:

```
ValueError: invalid literal for int()
```

Debe ver:

```
Edad inválida. Ingrese un número entero.
```

Separar:

> Error técnico vs mensaje de usuario

---




# PARTE II — DISEÑO DE SOFTWARE ROBUSTO


---

## Capas de manejo de errores

| Capa            | Responsabilidad    |
| --------------- | ------------------ |
| Infraestructura | detecta error real |
| Repositorio     | traduce error      |
| Servicio        | decide acción      |
| Presentación    | mensaje limpio     |

---

Ejemplo flujo:

```
Archivo corrupto → ValueError
Repositorio → DataFormatError
Servicio → decide abortar carga
UI → "El archivo contiene datos inválidos"
```

---

## Principio fundamental

> Solo la capa superior habla con el usuario

---



# PARTE III — IMPLEMENTACIÓN GUIADA

---


In [ ]:
class EdadInvalidaError(Exception):
    pass

class EdadNegativaError(Exception):
    pass

try:
    entrada = input("Edad: ")

    if not entrada.isdigit():
        raise EdadInvalidaError("La edad debe ser un número entero.")

    edad = int(entrada)

    if edad < 0:
        raise EdadNegativaError("La edad no puede ser negativa.")

except EdadInvalidaError as e:
    print(f"Error de formato: {e}")

except EdadNegativaError as e:
    print(f"Error lógico: {e}")

except Exception as e:
    print(f"Error inesperado: {e}")

else:
    print(f"Edad válida: {edad}")

finally:
    print("Fin del proceso.")

# Proyecto de clases:

## Arquitectura por Capas

```
app/
│
├── domain/
│   ├── exceptions.py
│   └── models.py
│
├── infrastructure/
│   └── file_repository.py
│
├── application/
│   └── user_service.py
│
└── main.py

```

---

# CAPA DOMAIN

## `domain/exceptions.py`



In [ ]:
# Excepciones de dominio (negocio)

class DomainError(Exception):
    """Base para errores de negocio."""
    pass


class EdadInvalidaError(DomainError):
    """Edad no numérica o formato inválido."""
    pass


class EdadNegativaError(DomainError):
    """Edad menor que 0."""
    pass


class LineaCorruptaError(DomainError):
    """Línea con formato incorrecto."""
    pass


class ArchivoVacioError(DomainError):
    """Archivo sin contenido."""
    pass


---

## `domain/models.py`



In [ ]:
from domain.exceptions import EdadNegativaError


# Modelo de dominio

class Usuario:

    def __init__(self, nombre: str, edad: int):
        if edad < 0:
            raise EdadNegativaError("La edad no puede ser negativa.")

        self.nombre = nombre
        self.edad = edad


---

# CAPA INFRASTRUCTURE

## `infrastructure/file_repository.py`

Responsable únicamente de leer archivos.



In [ ]:
import csv
from domain.exceptions import (
    LineaCorruptaError,
    ArchivoVacioError,
    EdadInvalidaError,
)
from domain.models import Usuario


class FileRepository:

    def __init__(self, path: str):
        self.path = path

    def read_users(self):
        usuarios = []

        try:
            with open(self.path, "r", encoding="utf-8", newline="") as f:
                reader = csv.DictReader(f)

                # Si no hay cabecera ni filas
                if reader.fieldnames is None:
                    raise ArchivoVacioError("El archivo está vacío.")

                for numero_linea, row in enumerate(reader, start=2):
                    # start=2 porque la cabecera es la línea 1
                    try:
                        if row is None:
                            raise LineaCorruptaError(
                                f"Línea corrupta en línea {numero_linea}"
                            )

                        nombre = (row.get("nombre") or "").strip()
                        edad_str = (row.get("edad") or "").strip()

                        if not nombre or not edad_str:
                            raise LineaCorruptaError(
                                f"Línea corrupta en línea {numero_linea}"
                            )

                        if not edad_str.isdigit():
                            raise EdadInvalidaError(
                                f"Edad inválida en línea {numero_linea}"
                            )

                        edad = int(edad_str)

                        usuario = Usuario(nombre, edad)
                        usuarios.append(usuario)

                    except ValueError:
                        raise LineaCorruptaError(
                            f"Línea corrupta en línea {numero_linea}"
                        )

                if not usuarios:
                    raise ArchivoVacioError("El archivo no contiene usuarios.")

        except FileNotFoundError:
            raise FileNotFoundError("El archivo no existe.")

        except PermissionError:
            raise PermissionError("Permisos insuficientes para leer el archivo.")

        except UnicodeDecodeError:
            raise UnicodeDecodeError(
                "utf-8",
                b"",
                0,
                1,
                "Encoding incorrecto."
            )

        return usuarios


---

# CAPA APPLICATION

## `application/user_service.py`

Orquesta la lógica.



In [ ]:
from infrastructure.file_repository import FileRepository


class UserService:

    def __init__(self, repository: FileRepository):
        self.repository = repository

    def procesar(self):
        return self.repository.read_users()


---

# CAPA PRESENTATION (ENTRYPOINT)

## `main.py`

Aquí usamos manejo profesional de excepciones.



In [ ]:
from infrastructure.file_repository import FileRepository
from application.user_service import UserService
from domain.exceptions import (
    ArchivoVacioError,
    LineaCorruptaError,
    EdadInvalidaError,
    EdadNegativaError,
)

def main():

    repo = FileRepository("usuarios.csv")
    service = UserService(repo)

    try:
        usuarios = service.procesar()

    except FileNotFoundError as e:
        print(f"[ERROR SISTEMA] {e}")

    except PermissionError as e:
        print(f"[ERROR PERMISOS] {e}")

    except UnicodeDecodeError:
        print("[ERROR ENCODING] El archivo no está en UTF-8.")

    except ArchivoVacioError as e:
        print(f"[ERROR DATOS] {e}")

    except LineaCorruptaError as e:
        print(f"[ERROR FORMATO] {e}")

    except EdadInvalidaError as e:
        print(f"[ERROR VALIDACIÓN] {e}")

    except EdadNegativaError as e:
        print(f"[ERROR NEGOCIO] {e}")

    except Exception as e:
        print(f"[ERROR INESPERADO] {e}")

    else:
        print("Usuarios cargados correctamente:")
        for u in usuarios:
            print(f"{u.nombre} - {u.edad}")

    finally:
        print("Proceso finalizado.")


if __name__ == "__main__":
    main()


# Arquitectura Hexagonal (Ports & Adapters)

También llamada:

> **Ports and Adapters Architecture**

Su objetivo es:

> Aislar completamente el dominio del mundo exterior (DB, API, archivos, frameworks).

---

## 🔹Idea central

El **núcleo de la aplicación** es el dominio.
Todo lo externo se conecta a través de **puertos (interfaces)**.

Visualmente:

```
        ┌───────────────┐
        │   API REST    │
        └──────┬────────┘
               │
        ┌──────▼────────┐
        │   ADAPTER     │
        └──────┬────────┘
               │
        ┌──────▼────────┐
        │    PORT       │  ← interfaz
        └──────┬────────┘
               │
        ┌──────▼────────┐
        │   DOMINIO     │  ← núcleo
        └───────────────┘
```

---

## 🔹 Componentes

### 1️⃣ Dominio

* Entidades
* Value Objects
* Reglas de negocio
* Excepciones

No depende de nada externo.

---

### 2️⃣ Puertos (Ports)

Son **interfaces** que el dominio define.

Ejemplo:

```python
class UserRepositoryPort:
    def save(self, user):
        pass
```

El dominio dice:

> “Necesito guardar usuarios, pero no me importa cómo”.

---

### 3️⃣ Adaptadores (Adapters)

Implementan los puertos.

Ejemplo:

```python
class FileUserRepository(UserRepositoryPort):
    ...
```

o

```python
class PostgresUserRepository(UserRepositoryPort):
    ...
```

El dominio no sabe cuál se usa.

---

## 🔹 Beneficios

* Bajo acoplamiento
* Alta testabilidad
* Independencia de frameworks
* Cambio de base de datos sin tocar negocio
* Cambio de API sin romper dominio

---

# 🧠 DDD Light

DDD = **Domain-Driven Design**

DDD completo es complejo.
DDD Light significa aplicar solo lo esencial:

---

## 🔹 Elementos clave

### 1️⃣ Entidades

Objetos con identidad.

```python
class Usuario:
    def __init__(self, id, nombre):
        self.id = id
        self.nombre = nombre
```

---

### 2️⃣ Value Objects

Objetos inmutables sin identidad.

```python
class Edad:
    def __init__(self, valor):
        if valor < 0:
            raise ValueError
        self.valor = valor
```

---

### 3️⃣ Servicios de Dominio

Cuando una lógica no pertenece a una entidad concreta.

---

### 4️⃣ Repositorios (como interfaz)

Abstracción de persistencia.

---

# 🎯 Relación entre Hexagonal y DDD

Hexagonal = estructura arquitectónica
DDD = enfoque de modelado del dominio

Juntas producen:

> Dominio fuerte + infraestructura intercambiable

---

# 🔬 Diferencia con arquitectura tradicional en capas

### Tradicional

```
Controller
   ↓
Service
   ↓
Repository
   ↓
Database
```

Problema:

* Dominio termina dependiendo del ORM.
* Lógica se dispersa.

---

### Hexagonal

```
Infraestructura → Adaptador → Puerto → Dominio
```

El dominio nunca depende hacia afuera.

---

# 🧩 Cuándo usar DDD Light + Hexagonal

✔ Sistemas medianos/grandes
✔ Lógica de negocio compleja
✔ Necesidad de test unitarios reales
✔ Posible cambio de infraestructura

No es necesario para:

* Scripts pequeños
* CRUD simples

---

# 🧠 En una frase

> Hexagonal protege el dominio.
> DDD modela correctamente el dominio.

